# KoHRM-Text-1.4B Colab T4 Long Knowledge Probe

This notebook checks a pretraining checkpoint's long-form knowledge signal after PT-style training by generating long responses from training-format instructions.

It follows the same training data layout used by `scripts/prepare_sft_data.py`:

```text
instruction text -> response text
```

The runtime wraps each instruction exactly in the HRM-Text control format:

```text
<|im_start|><|object_ref_start|>instruction<|im_end|>
```

`<|object_ref_start|>` is the `direct` condition. The model then generates the response until `<|box_end|>` or the token budget. The prompts below are intentionally close to the actual Korean legal raw corpus, Korean wiki raw corpus, BCAI finance QA, and terminal conversation builders used in pretraining.


## 1. Install Dependencies

This path intentionally avoids `transformers`, `AutoTokenizer`, and `AutoModelForCausalLM`. The current public export is a custom HRM-Text architecture, so the notebook uses the lightweight project helper instead.


In [ ]:
!pip -q install -U huggingface_hub hf_transfer safetensors
!pip -q install --force-reinstall -q "tokenizers>=0.22.0,<0.23.1"

## 2. Runtime Settings

`MAX_SEQ_LEN=1536` and long generation are intended for T4 knowledge probing. If Colab runs out of memory, lower `MAX_SEQ_LEN` to `1024` and `DEFAULT_MAX_NEW_TOKENS` to `256`.


In [ ]:
import os
import json
import gc
import importlib.util
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO_ID = "LLM-OS-Models/KoHRM-Text-1.4B"
REVISION = "main"
LOCAL_DIR = Path("/content/KoHRM-Text-1.4B")
HELPER_PATH = LOCAL_DIR / "kohrm_colab_generate.py"

MAX_SEQ_LEN = 1536
DEFAULT_MAX_NEW_TOKENS = 384
DEFAULT_MIN_NEW_TOKENS = 160

LONG_TEXT_SETTINGS = {
    "max_seq_len": MAX_SEQ_LEN,
    "temperature": 0.65,
    "top_p": 0.92,
    "repetition_penalty": 1.05,
    "no_repeat_ngram_size": 0,
    "condition": "direct",
}

print("repo:", REPO_ID)
print("revision:", REVISION)
print("local_dir:", LOCAL_DIR)
print("max_seq_len:", MAX_SEQ_LEN)
print("default max/min new tokens:", DEFAULT_MAX_NEW_TOKENS, DEFAULT_MIN_NEW_TOKENS)

## 3. Download Latest Public Checkpoint

The notebook downloads only the files needed for public `model.safetensors` inference and the helper runtime. If the helper is missing from the model repo, it falls back to the GitHub repository.


In [ ]:
from huggingface_hub import snapshot_download

LOCAL_DIR.mkdir(parents=True, exist_ok=True)
patterns = [
    "README.md",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "model.safetensors",
    "kohrm_colab_generate.py",
    "notebooks/kohrm_colab_generate.py",
]

snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    revision=REVISION,
    local_dir=str(LOCAL_DIR),
    local_dir_use_symlinks=False,
    allow_patterns=patterns,
)

nested_helper = LOCAL_DIR / "notebooks" / "kohrm_colab_generate.py"
if not HELPER_PATH.exists() and nested_helper.exists():
    HELPER_PATH.write_text(nested_helper.read_text(encoding="utf-8"), encoding="utf-8")

if not HELPER_PATH.exists():
    repo = Path("/content/KoHRM-text")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/LLM-OS-Models/KoHRM-text", str(repo)],
            check=True,
        )
    HELPER_PATH.write_text((repo / "notebooks" / "kohrm_colab_generate.py").read_text(encoding="utf-8"), encoding="utf-8")

for name in ["config.json", "tokenizer.json", "model.safetensors", "README.md", "kohrm_colab_generate.py"]:
    p = LOCAL_DIR / name
    print(name, "OK" if p.exists() else "MISSING", p)

## 4. Inspect Config and Training Wrapper

This cell checks the model shape and special tokens. The prompt wrapper shown here is the exact wrapper used for all probes below.


In [ ]:
spec = importlib.util.spec_from_file_location("kohrm_colab_generate", HELPER_PATH)
kohrm = importlib.util.module_from_spec(spec)
sys.modules["kohrm_colab_generate"] = kohrm
spec.loader.exec_module(kohrm)

config = json.loads((LOCAL_DIR / "config.json").read_text(encoding="utf-8"))
print(json.dumps({
    "model_type": config.get("model_type"),
    "architectures": config.get("architectures"),
    "vocab_size": config.get("vocab_size"),
    "hidden_size": config.get("hidden_size"),
    "num_hidden_layers": config.get("num_hidden_layers"),
    "num_attention_heads": config.get("num_attention_heads"),
    "H_cycles": config.get("H_cycles"),
    "L_cycles": config.get("L_cycles"),
    "max_position_embeddings": config.get("max_position_embeddings"),
    "prefix_lm": config.get("prefix_lm"),
}, indent=2, ensure_ascii=False))

from tokenizers import Tokenizer
raw_tok = Tokenizer.from_file(str(LOCAL_DIR / "tokenizer.json"))
for token in ["<|im_start|>", "<|object_ref_start|>", "<|object_ref_end|>", "<|quad_start|>", "<|quad_end|>", "<|im_end|>", "<|box_end|>"]:
    print(f"{token:22s}", raw_tok.token_to_id(token))

example_instruction = "비씨카드는 어떤 회사인가요?"
print("direct condition token:", kohrm.condition_to_tokens("direct"))
print("wrapped prompt:", kohrm.format_kohrm_prompt(example_instruction, condition="direct"))

## 5. Load Model Once

Loading can take a few minutes on Colab. The helper uses PyTorch scaled-dot-product attention and a static KV cache. It is slower than the training-time FlashAttention path, but it is enough for long text inspection.


In [ ]:
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(torch.cuda.get_device_name(0))

model, tokenizer, cfg = kohrm.load_kohrm(LOCAL_DIR, max_gpu_memory_gib=14.0)
print("loaded dtype:", next(model.parameters()).dtype)
print("loaded device:", next(model.parameters()).device)
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free/total GiB after load: {free / 2**30:.2f}/{total / 2**30:.2f}")

## 6. PT-Format Long Knowledge Prompts

These long-response probes use prompts shaped like the data that went into pretraining:

- BCAI Finance Kor: plain Korean finance QA instruction.
- Korean wiki raw corpus: `다음은 한국어 위키백과 문서 원문 일부입니다...` instruction.
- Korean legal raw corpus: `다음은 대한민국 법령/자치법규 원문 일부입니다...` instruction.
- Terminal conversation corpus: `다음 터미널/코딩 작업 대화 맥락에서...` instruction.

The notebook prints output length and the raw generated text. Judge the text manually: domain terms, Korean fluency, factual continuity, repetition, and whether it collapses to one-token answers.


In [ ]:
KNOWLEDGE_PROMPTS = [
    {
        "name": "finance_bc_card_company",
        "source_format": "BCAI Finance Kor plain QA",
        "prompt": "비씨카드는 어떤 회사인가요?",
        "max_new_tokens": 420,
        "min_new_tokens": 180,
    },
    {
        "name": "finance_exchange_investment",
        "source_format": "BCAI Finance Kor plain QA",
        "prompt": "환율 변동이 개인 투자에 미치는 영향과 대비 전략은 무엇인가요?",
        "max_new_tokens": 420,
        "min_new_tokens": 180,
    },
    {
        "name": "kowiki_hunminjeongeum_raw_style",
        "source_format": "kowiki raw instruction style",
        "prompt": """다음은 한국어 위키백과 문서 원문 일부입니다. 백과사전식 한국어, 고유명사, 날짜, 기술/사회/문화 지식을 그대로 학습하십시오.

[문서명]
훈민정음

[부분]
1/1""",
        "max_new_tokens": 520,
        "min_new_tokens": 220,
    },
    {
        "name": "kowiki_jimmy_carter_raw_style",
        "source_format": "kowiki raw instruction style",
        "prompt": """다음은 한국어 위키백과 문서 원문 일부입니다. 백과사전식 한국어, 고유명사, 날짜, 기술/사회/문화 지식을 그대로 학습하십시오.

[문서명]
지미 카터

[부분]
1/3""",
        "max_new_tokens": 520,
        "min_new_tokens": 220,
    },
    {
        "name": "legal_criminal_code_raw_style",
        "source_format": "korean legal raw instruction style",
        "prompt": """다음은 대한민국 법령/자치법규 원문 일부입니다. 법률 한국어, 조문 구조, 번호 체계, 기관명, 시행일자 표현을 그대로 학습하십시오.

[자료종류]
law

[문서명]
형법

[경로]
kr/형법/법률.md

[부분]
1/1""",
        "max_new_tokens": 520,
        "min_new_tokens": 220,
    },
    {
        "name": "legal_restoration_raw_style",
        "source_format": "korean legal raw instruction style",
        "prompt": """다음은 대한민국 법령/자치법규 원문 일부입니다. 법률 한국어, 조문 구조, 번호 체계, 기관명, 시행일자 표현을 그대로 학습하십시오.

[자료종류]
law

[문서명]
10ㆍ27법난 피해자의 명예회복 등에 관한 법률

[경로]
kr/10ㆍ27법난피해자의명예회복등에관한법률/법률.md

[부분]
1/1""",
        "max_new_tokens": 520,
        "min_new_tokens": 220,
    },
    {
        "name": "terminal_conversation_style_long",
        "source_format": "local terminal conversation instruction style",
        "prompt": """다음 터미널/코딩 작업 대화 맥락에서 assistant가 이어서 수행할 분석, 계획, 명령 JSON 또는 최종 응답을 작성하십시오.

[user]
현재 디렉터리에서 용량이 큰 파일을 찾아 디스크 정리 후보를 보고 싶습니다. 숨김 폴더와 일반 폴더를 모두 확인하되, 결과는 사람이 읽기 쉽게 정리해 주세요.""",
        "max_new_tokens": 420,
        "min_new_tokens": 160,
    },
]

print("probe count:", len(KNOWLEDGE_PROMPTS))
for item in KNOWLEDGE_PROMPTS:
    print("-", item["name"], "|", item["source_format"], "| max/min", item["max_new_tokens"], item["min_new_tokens"])


## 7. Run Long Generations

The goal is to inspect the generated text itself after PT-style training: length, Korean fluency, domain terms, factual continuity, and repetition.


In [ ]:
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False).ids)

RUN_ONLY = None  # Example: {"finance_bc_card_company", "kowiki_hunminjeongeum_raw_style"}

results = []
for case in KNOWLEDGE_PROMPTS:
    if RUN_ONLY is not None and case["name"] not in RUN_ONLY:
        continue

    prompt = case["prompt"]
    max_new = case.get("max_new_tokens", DEFAULT_MAX_NEW_TOKENS)
    min_new = case.get("min_new_tokens", DEFAULT_MIN_NEW_TOKENS)

    print("=" * 100)
    print("case:", case["name"])
    print("source_format:", case["source_format"])
    print("prompt_chars:", len(prompt), "prompt_tokens:", count_tokens(kohrm.format_kohrm_prompt(prompt, condition="direct")))
    print("max_new_tokens:", max_new, "min_new_tokens:", min_new)
    print("--- prompt ---")
    print(prompt)
    print("--- output ---")

    output = kohrm.generate_from_loaded(
        model,
        tokenizer,
        cfg,
        prompt,
        max_new_tokens=max_new,
        min_new_tokens=min_new,
        **LONG_TEXT_SETTINGS,
    )
    out_tokens = count_tokens(output)
    results.append({
        "case": case["name"],
        "source_format": case["source_format"],
        "output_chars": len(output),
        "output_tokens": out_tokens,
        "output": output,
    })
    print(output)
    print("--- output stats ---")
    print("chars:", len(output), "tokens:", out_tokens)

print("=" * 100)
print(json.dumps([{k: r[k] for k in ["case", "source_format", "output_chars", "output_tokens"]} for r in results], indent=2, ensure_ascii=False))


## 8. Optional Decode Settings Sweep

Use this only if long output collapses to short tokens or repeats. It keeps the same training-format prompts and changes only decoding parameters.


In [ ]:
SWEEP_CASE_NAME = "finance_exchange_investment"
SWEEP_PROMPT = next(item["prompt"] for item in KNOWLEDGE_PROMPTS if item["name"] == SWEEP_CASE_NAME)

SWEEP_SETTINGS = [
    {"name": "deterministic_minlen", "temperature": 0.0, "top_p": 1.0, "repetition_penalty": 1.02, "no_repeat_ngram_size": 0},
    {"name": "sample_balanced", "temperature": 0.65, "top_p": 0.92, "repetition_penalty": 1.05, "no_repeat_ngram_size": 0},
    {"name": "sample_more_diverse", "temperature": 0.85, "top_p": 0.95, "repetition_penalty": 1.08, "no_repeat_ngram_size": 0},
]

for settings in SWEEP_SETTINGS:
    run_settings = dict(LONG_TEXT_SETTINGS)
    run_settings.update({k: v for k, v in settings.items() if k != "name"})
    print("=" * 100)
    print("decode:", settings["name"])
    output = kohrm.generate_from_loaded(
        model,
        tokenizer,
        cfg,
        SWEEP_PROMPT,
        max_new_tokens=320,
        min_new_tokens=120,
        **run_settings,
    )
    print(output)
    print("chars:", len(output), "tokens:", count_tokens(output))


## 9. How To Read Results

For this PT knowledge probe, useful signs are:

- The output is long enough to inspect, not `Yes`, `B`, `0`, or one short fragment.
- Korean legal/wiki/finance outputs contain domain terms and coherent Korean paragraphs.
- Raw-style prompts may produce article/law-like continuations because that is how raw corpora were converted for PT.
- Terminal conversation style may produce analysis plus commands, because that is what the local terminal conversation data used as the response target.

If every long prompt still collapses to one-token or unrelated boilerplate despite `min_new_tokens`, the next thing to debug is the public converted weight/runtime compatibility, not prompt wording.
